# Dự báo giá nhà — Notebook Basic

Bản ngắn vẫn giữ đúng Development → MODEL LOCK → Test.

In [1]:
from pathlib import Path
import hashlib, json, sys, pandas as pd, numpy as np
ROOT = Path.cwd() if (Path.cwd()/"data").exists() else Path.cwd().parent
if str(ROOT/"src") not in sys.path: sys.path.insert(0, str(ROOT/"src"))
from house_price.data import load_house_data, split_xy, sha256_file
print("PROJECT_ROOT: <project root detected>")

PROJECT_ROOT: <project root detected>


## Phase A — Development

In [2]:
tr = load_house_data("house_train_80.csv"); va = load_house_data("house_validation_10.csv")
Xtr, ytr = split_xy(tr); Xva, yva = split_xy(va)
assert tr.shape == (1168, 81) and va.shape == (146, 81)
print("Development only:", Xtr.shape, Xva.shape)

Development only: (1168, 79) (146, 79)


In [3]:
summary_path = ROOT/"outputs"/"summary.json"
policy_path = ROOT/"outputs"/"model_selection_policy.json"
fingerprint_path = ROOT/"outputs"/"source_fingerprint.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
policy = json.loads(policy_path.read_text(encoding="utf-8"))
fingerprint = json.loads(fingerprint_path.read_text(encoding="utf-8"))
issues = []
# Data freshness uses file checksums only; test target values are not loaded here.
for item in summary.get("data_hashes", []):
    path = ROOT/item["file"]
    if sha256_file(path) != item["sha256"]: issues.append(f"data hash mismatch: {item['file']}")
model_path = ROOT/summary["artifact"]["path"]
if sha256_file(model_path) != summary["artifact"]["model_sha256"]: issues.append("artifact hash mismatch")
if summary["selected_model"] != policy["locked_model"]: issues.append("selected model mismatch")
if summary["selected_params"] != policy["locked_params"]: issues.append("selected params mismatch")
for rel, expected in fingerprint.get("files", {}).items():
    if sha256_file(ROOT/rel) != expected: issues.append(f"source fingerprint mismatch: {rel}")
if not summary.get("generated_at_utc") or not fingerprint.get("generated_at_utc"): issues.append("missing generated_at timestamp")
if issues:
    raise RuntimeError("STALE EVIDENCE: " + "; ".join(issues) + ". Regenerate canonical evidence before using this notebook.")
print("FRESHNESS: PASS")
print("summary generated_at:", summary["generated_at_utc"])
print("fingerprint generated_at:", fingerprint["generated_at_utc"])

FRESHNESS: PASS
summary generated_at: 2026-08-26T11:51:25.286277+00:00
fingerprint generated_at: 2026-08-26T17:29:25.344975+00:00


## Phase B — MODEL LOCK

In [4]:
print("Locked model:", summary["selected_model"])
print("Locked params:", summary["selected_params"])
assert policy["locked_model"] == summary["selected_model"]
assert policy["locked_params"] == summary["selected_params"]
assert policy["uses_test"] is False
print("MODEL LOCKED")

Locked model: ElasticNet
Locked params: {'regressor__model__l1_ratio': 0.2, 'regressor__model__alpha': 0.001}
MODEL LOCKED


## Phase C — Test

In [5]:
te = load_house_data("house_test_10.csv")
from house_price.validation import validate_splits
print(validate_splits(tr, va, te))
Xte, yte = split_xy(te)
print("TEST UNLOCKED AFTER MODEL LOCK:", Xte.shape)

{'shapes': {'train': (1168, 81), 'validation': (146, 81), 'test': (146, 81)}, 'unique_ids': 1460, 'target': 'SalePrice', 'same_schema': True, 'duplicate_rows': {'train': 0, 'validation': 0, 'test': 0}}
TEST UNLOCKED AFTER MODEL LOCK: (146, 79)


In [6]:
from house_price.modeling import build_end_to_end_estimator, model_definitions
from house_price.evaluation import regression_metrics
base = model_definitions()[summary["selected_model"]]
model = build_end_to_end_estimator(Xtr, base)
model.set_params(**summary["selected_params"])
model.fit(pd.concat([Xtr, Xva], ignore_index=True), pd.concat([ytr, yva], ignore_index=True))
pred = model.predict(Xte)
metrics = regression_metrics(yte, pred)
metrics

{'RMSE': 22228.6030944359,
 'MAE': 15452.339433784216,
 'R2': 0.9076281882189048,
 'RMSLE': 0.11772471181677151}

In [7]:
for key, value in summary["test_metrics"].items():
    assert abs(metrics[key] - value) < 1e-6
print("PASS: Basic reproduces canonical metrics")

PASS: Basic reproduces canonical metrics


In [8]:
import joblib, tempfile
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp)/"basic_model.joblib"
    joblib.dump(model, path); loaded = joblib.load(path)
    assert np.allclose(loaded.predict(Xte.head(3)), pred[:3])
print("PASS: temporary serialization only; canonical artifact unchanged")

PASS: temporary serialization only; canonical artifact unchanged


## Kết luận
Basic notebook không mở `y_test` trước model lock và tái tạo đúng canonical metrics.